This notebooks aim is to answer the question, which terms actually exist in the picrust2 results of the data. Is there more terms that we are missing? and How to group these terms

# 1. Imports and Data preparation

In [5]:
import os
import sys
from pathlib import Path
import pickle
import pyarrow

# Data processing and analysis
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
from typing import Set, List, Union, Dict, Tuple, Optional

In [6]:

sys.path.append(os.path.abspath('..'))  # Ensures the project root is in Python's search path

if Path("/kaggle").exists():
    
    # Create directory structure
    !mkdir -p corrosion_scoring
    
    # Download the necessary files each session always
    !wget -O corrosion_scoring/__init__.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/__init__.py
    !wget -O corrosion_scoring/global_terms.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/global_terms.py
    !wget -O corrosion_scoring/scoring_system.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/scoring_system.py
    !wget -O corrosion_scoring/term_processor.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/term_processor.py
    # Add current directory to path
    import sys
    sys.path.append(os.getcwd())
    
    # Import package
    import corrosion_scoring as cs
else:
    print("Running in local (VSCode) environment")
    
    ## When in vscode local env first time only
    #!pip install git+https://github.com/MagicAlex238/2_Micro.git#subdirectory=corrosion_scoring_root
    import corrosion_scoring as cs

Running in local (VSCode) environment


In [7]:
# environment check
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    # For Kaggle # Whole filtered Data
    base_dir =  Path("/kaggle/input/")
    eccontri_path = base_dir / "/eccontri-uniprot-enriched/ECcontri_Uniprot_enriched.parquet"
    pathway_path = base_dir / "gterms/pathways.csv"
    react_path = base_dir / "gterms/reactions.csv"
    # Directory to output large files 
    large_dir =  Path("/kaggle/working/")
    # Directory to output large files # eccontris, compilated db

else:              
    # For Vscode # Whole filtered Data large size dir for large files hosted instead in kaggle
    large_dir = Path("/home/beatriz/MIC")
    # Directory to output large files # eccontris, compilated dbs
    output_large = large_dir / "output_large"
    # Whole filtered Data
    eccontri_path = output_large / 'ECcontri_Uniprot_enriched.parquet'
    pathway_path = large_dir / "2_Micro/data_picrust/pathways.csv"
    react_path = large_dir / "2_Micro/data_picrust/reactions.csv"

In [8]:
# Whole filtered Data
ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)

# 2. Validating terms as real_terms
## 2.1. Checking the terms agains the teoretical global_terms

In [9]:
def validate_terms(df, global_terms_list):
    """
    Checks which terms from a list of global dictionaries exist in the data.
    
    Args: df : dataframe with data to check
          global_terms: global list of dictionaries with metal_terms, pathways, mechanisms, functional categories, organic proceses and keywords
    
    Returns:  dict: {category: [found_terms]}
    """
    df = df.copy()
    found = {}
    cols_terms = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
            'corrosion_mechanisms', 'functional_categories', 
            'corrosion_keyword_groups', 'corrosion_synergies', 'organic_processes'
        ]

    for d in global_terms_list:
        for category, terms in d.items():
            if isinstance(terms, dict):
                # Handle functional_categories special case, nested with scores
                if 'terms' in terms and 'score' in terms:
                    # This is functional_categories format: {'terms': [...], 'score': 1.5}
                    existing = []
                    for term in terms['terms']:  # Access the 'terms' key
                        for col in cols_terms:
                            if col in df.columns:
                                if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                    existing.append(term)
                                    break
                    if existing:
                        found[category] = existing
                else:
                    # Handle other nested dictionaries
                    for subcategory, subterms in terms.items():
                        if isinstance(subterms, list):  # Making sure it's a list
                            existing = []
                            for term in subterms:
                                for col in cols_terms:
                                    if col in df.columns:
                                        if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                            existing.append(term)
                                            break
                            if existing:
                                found[f"{category}.{subcategory}"] = existing
            else:
                # Handle simple lists
                existing = []
                for term in terms:
                    for col in cols_terms:
                        if col in df.columns:
                            if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                existing.append(term)
                                break
                if existing:
                    found[category] = existing
    
    return found
#sample= ECcontri_Uniprot_enriched.sample(n=15000)

In [10]:
sample =ECcontri_Uniprot_enriched.sample(n=15000)

In [11]:
real_terms = validate_terms(sample,
    [cs.metal_terms,
    cs.corrosion_mechanisms,
    cs.pathway_categories,
    cs.organic_categories,
    cs.corrosion_synergies,
    cs.functional_categories,
    cs.corrosion_keyword_groups
])

In [12]:
# path to the list of dictionaries
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    rt_path = large_dir / 'real_terms.pkl'
    gterms_path = large_dir / 'real_pathways_reactions.pkl'
else:
    rt_path = output_large / 'real_terms.pkl'
    gterms_path = output_large / 'real_pathways_reactions.pkl'

In [13]:
# Saving the new dataframe
#with open(rt_path, 'wb') as f:
#    pickle.dump(real_terms, f)

In [14]:
# Reading the dictionaries
with open(rt_path, 'rb') as f:
    real_terms= pickle.load(f)
# Print in compact format
for category, terms in real_terms.items():
    terms_str = ', '.join(terms)
    print(f"'{category}': [{terms_str}]")

'iron': [iron, ferric, heme, iron-sulfur, siderophore, ferritin, ferredoxin, rubredoxin]
'manganese': [mn]
'copper': [copper]
'nickel': [Ni2+]
'cobalt': [cobalt, cobalamin, vitamin B12]
'magnesium': [magnesium]
'calcium': [Ca2+, calcium]
'Mo': [Mo, molybdenum, molybdopterin, molybdenum cofactor]
'V5+': [V5+, vanadium]
'Al3+': [Al3+]
'Cr3+': [Cr3+]
'zinc': [Zn2+, zinc]
'selenium': [selenium, Se, selenocysteine, selenoprotein, selenite, selenate]
'lead': [lead]
'arsenic': [arsenic, arsenate]
'mercury': [mercury, mercuric]
'phosphate': [phosphate, orthophosphate]
'nitrate': [NO3-, nitrate]
'nitrite': [nitrite]
'chloride': [Cl-, chloride]
'sulfate': [sulfate]
'sulfide': [sulfide, h2s]
'thiosulfate': [thiosulfate]
'oxygen': [O2, oxygen, oxidase, superoxide, peroxide]
'hydrogen': [hydrogenase, h2]
'organics': [methane, methane, methanogenesis, formate, formate, formic acid, acetate, acetate, acetic acid, propionate, propionate, propionic acid, butyrate, butyrate, butyric acid, lactate, lacti

## 2.2 Critical review of real terms agains global terms

The process undergone for the scoring system has been iterative and during the first iteration it was noticed that pathways and mechanisms are highly interconnected due to the fact that one bacterium expresses multiple proteins across many pathways and mechanisms. Pathway and mechanism categories exhibit substantial overlap due to multi-protein expression patterns within individual bacterial strains. As a response to this iteration, functional_categories were designed to reconcile this complexity by grouping related processes and make it more about functional metabolism. On a following iteration more modern terms were introduced and diverse terms were tried. Ultimately, it was evident that it was necesary a reality check to validate the terms with the bioinformatics annotations. 
An script was done to critically evaluate the real terms possible to be mined from the compiled database after the enrichment of the data (ECcontri_Uniprot_enriched) with ec_records. The script compared the enriched data with the global terms which teoretically proposed the dictionaries grouped by categories namely: metal_terms, mechanisms, pathways, functional_categories, organic_processes, synergies and keywords.
Analysis of the enriched data's real terms, prompt to redefine the corrosion scoring system by eliminating non-existent terms and consolidating overlapping categorical structures. Yet some theoretical terms are left for teoretical completness. A manual curation was done to reasign categories for efficient computational resource allocation.
The categories to consolidate are: corrosion_synergies,metal_terms, functional_categories, mechanisms and pathways. The categories to remove are: corrosion_keyword_groups and organic_processes. A hierarchy of the categories is stablished, this allows a first term algorithm to prioritize the terms allocated, in order to prevent duplicates.
The scoring takes into account only corrosion_synergies,metal_terms and functional_categories.


# 2.3 Finding new real_terms : class BiologicalTermDiscovery
Before the terms are compile and arrange on different dictionaries, it was created a script to assest the possible new terms that could be in the enriched df. By using a comprehensive toolkit for discovering and analyzing biological terms from datasets. This class provides methods to extract terms from text data, identify patterns,and discover novel biological terms based on existing validated terms (real_terms).

In [19]:
from collections import Counter, defaultdict
import re
import pandas as pd
from typing import Dict, List, Set, Optional, Tuple

class BiologicalTermDiscovery:
    """
    A comprehensive toolkit for discovering and analyzing biological terms from datasets.
    """
    
    def __init__(self, min_term_length: int = 3, stop_words: Optional[Set[str]] = None):
        """
        Initialize the BiologicalTermDiscovery class.
        
        Args:
            min_term_length: Minimum length for extracted terms
            stop_words: Set of words to exclude from analysis
        """
        self.min_term_length = min_term_length
        self.stop_words = stop_words or {
            'and', 'or', 'the', 'of', 'in', 'to', 'for', 'with', 'by', 
            'from', 'at', 'on', 'high', 'general', 'families', 'viral', 
            'rna', 'gene', 'direct', 'organics', 'groups', 'ambiguous', 'messenger'
        }
    
    def extract_all_terms(self, series: pd.Series) -> Counter:
        """
        Extract all terms from a pandas Series containing text data.
        
        Args:
            series: Pandas Series containing text data
            
        Returns:
            Counter object with term frequencies
        """
        terms = Counter()
        
        for text in series.dropna():
            if isinstance(text, str):
                # Split on various separators while preserving meaningful phrases
                chunks = re.split(r'[,;|\n\t\s\(\)\[\]]+', text)
                
                for chunk in chunks:
                    # Clean but preserve meaningful separators
                    cleaned = re.sub(r'^[^\w]+|[^\w]+$', '', chunk.lower())
                    
                    # Filter terms based on length and stop words
                    if (len(cleaned) >= self.min_term_length and 
                        not cleaned.isdigit() and 
                        cleaned not in self.stop_words):
                        terms[cleaned] += 1
                        
                        # Extract individual words from compound terms
                        if '_' in cleaned or '-' in cleaned:
                            parts = re.split(r'[_-]+', cleaned)
                            for part in parts:
                                if (len(part) >= self.min_term_length and 
                                    not part.isdigit() and 
                                    part not in self.stop_words):
                                    terms[part] += 1
        
        return terms
    
    def create_pattern_dictionary(self, real_terms_dict: Dict[str, any]) -> Dict[str, List[str]]:
        """
        Create pattern dictionary from real terms to identify similar terms.
        
        Args:
            real_terms_dict: Dictionary mapping categories to lists of real terms
            
        Returns:
            Dictionary mapping patterns to example terms
        """
        all_terms = []
        
        # Handle multiple dictionaries
        if isinstance(real_terms_dict, dict) and any(isinstance(v, dict) and 'terms' in v for v in real_terms_dict.values()):
            # Single dictionary with nested structure (like functional_categories)
            for category, terms_data in real_terms_dict.items():
                if isinstance(terms_data, dict) and 'terms' in terms_data:
                    all_terms.extend(terms_data['terms'])
                elif isinstance(terms_data, list):
                    all_terms.extend(terms_data)
        else:
            # Simple dictionary or multiple dictionaries
            for category, terms_list in real_terms_dict.items():
                if isinstance(terms_list, list):
                    all_terms.extend(terms_list)
                elif isinstance(terms_list, dict):
                    for subcat, subterms in terms_list.items():
                        if isinstance(subterms, list):
                            all_terms.extend(subterms)
        
        pattern_groups = defaultdict(set)
        #  biologically relevant patterns
        biological_prefixes = {'fer', 'iron', 'sulf', 'thio', 'oxy', 'nitr', 'met', 'acet', 'prop', 'mang', 'chrom', 'PWY-', 'RXN-', 'superpathway'}
        enzyme_suffixes = {'*ase', '*tion', '*ate', '*ide', 'reductase', 'oxidase', 'dehydrogenase', 'transferase', 'isomerase'}
        
        
        for term in all_terms:
            if isinstance(term, str):
                term_lower = term.lower()
                
                # Extract meaningful prefixes
                if len(term_lower) >= 5:
                    for prefix_len in [3, 4, 5]:
                        if prefix_len < len(term_lower):
                            prefix = term_lower[:prefix_len]
                            if prefix in  biological_prefixes : {'fer', 'iron', 'sulf', 'thio', 'oxy', 'nitr', 'met', 'acet', 'prop', 'mang', 'chrom', 'PWY-', 'RXN-', 'superpathway'}
                            pattern_groups[prefix].add(term_lower)
                
                # Extract meaningful suffixes
                if len(term_lower) >= 6:
                    for suffix_len in [3, 4]:
                        suffix = term_lower[-suffix_len:]
                        if suffix in enzyme_suffixes : {'*ase', '*tion', '*ate', '*ide', 'reductase', 'oxidase', 'dehydrogenase', 'transferase', 'isomerase'}
                        pattern_groups[f"*{suffix}"].add(term_lower)
              
        
        # Keep only patterns with multiple matches
        return {
            pattern: list(terms) for pattern, terms in pattern_groups.items() 
            if len(terms) >= 2
        }
    
    def find_pattern_matches(self, all_terms: Counter, patterns: Dict[str, List[str]]) -> Dict[str, List[str]]:
        """
        Find terms that match established patterns but are not in the original real set.
        
        Args:
            all_terms: Counter of all extracted terms
            patterns: Dictionary of patterns to match against
            
        Returns:
            Dictionary mapping pattern types to lists of new terms
        """
        pattern_discoveries = defaultdict(list)
        
        for pattern, known_examples in patterns.items():
            known_set = set(known_examples)
            
            if pattern.startswith('*'):
                # Suffix pattern matching
                suffix = pattern[1:]
                for term in all_terms.keys():
                    if term.endswith(suffix) and term not in known_set:
                        pattern_discoveries[f"new_{suffix}_terms"].append(term)
            else:
                # Prefix pattern matching
                for term in all_terms.keys():
                    if term.startswith(pattern) and term not in known_set:
                        pattern_discoveries[f"new_{pattern}_variants"].append(term)
        
        return pattern_discoveries
    
    def discover_novel_terms(self, df: pd.DataFrame, 
                           metal_terms: Dict[str, List[str]],
                           corrosion_synergies: Dict[str, List[str]],
                           functional_categories: Dict[str, Dict],
                           text_columns: List[str],
                           min_frequency: int = 10) -> List[str]:
        """
        Main method to discover novel biological terms using pattern analysis.
        
        Args:
            df: DataFrame containing the biological data
            metal_terms: Dictionary of metal-related terms
            corrosion_synergies: Dictionary of corrosion synergy terms
            functional_categories: Dictionary of functional category terms
            text_columns: List of column names containing text data
            min_frequency: Minimum frequency threshold for terms
            
        Returns:
            List of new terms discovered
        """
        # Combine all real terms dictionaries
        all_real_dicts = {
            **metal_terms,
            **corrosion_synergies,
            **functional_categories
        }
        
        # Extract all terms from specified columns
        all_discovered_terms = Counter()
        
        for col in text_columns:
            if col in df.columns:
                column_terms = self.extract_all_terms(df[col])
                all_discovered_terms.update(column_terms)
        
        # Create pattern dictionary from real terms
        patterns = self.create_pattern_dictionary(all_real_dicts)
        
        #  biologically relevant patterns
        biological_prefixes = {'fer', 'iron', 'sulf', 'thio', 'oxy', 'nitr', 'met', 'acet', 'prop', 'mang', 'chrom', 'PWY-', 'RXN-', 'superpathway'}
        enzyme_suffixes = {'*ase', '*tion', '*ate', '*ide', 'reductase', 'oxidase', 'dehydrogenase', 'transferase', 'isomerase'}
        
        focused_patterns = {}
        for pattern, examples in patterns.items():
            if (any(pattern.startswith(prefix) for prefix in biological_prefixes) or
                any(pattern == suffix for suffix in enzyme_suffixes) or
                len(examples) >= 3):
                focused_patterns[pattern] = examples
        
        # Create flat set of all real terms
        all_real_terms = set()
        for terms_data in all_real_dicts.values():
            if isinstance(terms_data, list):
                all_real_terms.update(term.lower() for term in terms_data)
            elif isinstance(terms_data, dict):
                if 'terms' in terms_data:
                    all_real_terms.update(term.lower() for term in terms_data['terms'])
                else:
                    for subterms in terms_data.values():
                        if isinstance(subterms, list):
                            all_real_terms.update(term.lower() for term in subterms)
        
        # Find pattern-based discoveries
        pattern_matches = self.find_pattern_matches(all_discovered_terms, focused_patterns)
        
        # Filter by frequency and biological relevance
        new_terms = []
        for pattern, matches in pattern_matches.items():
            for term in matches:
                if (all_discovered_terms[term] >= min_frequency and 
                    not any(skip_word in term for skip_word in self.stop_words) and
                    not term.startswith('br:ko') and
                    not re.match(r'^[a-z]{2}:\w+\d+', term)):
                    new_terms.append(term)
        
        # Find completely novel high-frequency terms
        for term, freq in all_discovered_terms.most_common(100):
            if (freq >= min_frequency and
                term not in all_real_terms and
                len(term) >= 5 and
                not any(skip_word in term for skip_word in self.stop_words) and
                not term.startswith('br:ko') and
                not re.match(r'^[a-z]{2}:\w+\d+', term) and
                term not in new_terms):
                new_terms.append(term)
        
        # Remove duplicates and return sorted list
        return sorted(list(set(new_terms)))

In [17]:
metal_terms = {	
'iron': ['Fe2+', 'Fe3+', 'iron', 'ferrous', 'ferric', 'heme', 'iron-sulfur', 'rust', 'ochre', 'iron oxide', 'siderophore', 'ferritin'],	
'manganese': ['Mn2+',  'manganese', 'mn', 'manganous', 'manganic', 'manganese oxidation', 'manganese oxide', 'MnO2'],	
'copper': ['Cu+', 'Cu2+', 'copper', 'cupric', 'cuprous', 'copper oxide', 'copper corrosion'],	
'nickel': ['Ni2+', 'nickel', 'nickelous', 'nickel oxidation', 'nickel reduction'],	
'cobalt': ['Co2+',  'cobalt', 'cobaltous', 'cobalamin', 'vitamin B12'],	
'magnesium': ['Mg2+', 'magnesium', 'magnesium oxide'],	
'calcium': ['Ca2+', 'calcium', 'calcium carbonate', 'calcite', 'calcium precipitation'],	
'Mo': ['Mo',  'molybdenum', 'molybdopterin', 'molybdenum cofactor'],	
'V5+': ['V5+', 'vanadium', 'vanadate', 'vanadyl'],	
'Al3+': ['Al3+', 'aluminum', 'aluminate', 'aluminum oxide'],	
'Cr3+': ['Cr3+', 'Cr6+', 'chromium', 'chromate', 'dichromate', 'chromium oxide'],	
'zinc': ['Zn2+', 'zinc', 'zinc finger', 'zinc oxide'],	
'sodium': ['Na+', 'sodium', 'NaCl', 'sodium transport', 'sodium gradient'],	
'potassium': ['K+', 'potassium', 'KCl', 'potassium transport', 'potassium channel'],	
'selenium': ['selenium', 'Se', 'selenocysteine', 'selenoprotein', 'selenite'],	
'barium': ['Ba2+', 'barium', 'barium sulfate', 'barite'],	
'strontium': ['Sr2+', 'strontium', 'strontium carbonate', 'strontium sulfate'], 	
'lead': ['Pb2+', 'Pb4+', 'lead', 'plumbous', 'plumbic', 'lead oxide'], 	
'arsenic': ['As3+', 'As5+', 'arsenic', 'arsenite', 'arsenate', 'arsenic oxidation'], 	
'mercury': ['Hg2+', 'Hg+', 'mercury', 'mercuric', 'mercurous', 'mercury methylation'], 	
'phosphate': ['HPO4-2', 'PO4-3', 'phosphate', 'phosphates'],	
'nitrogen':['NO3-', 'nitrate', 'nitrates','NO2-', 'nitrite', 'nitrites'],	
'chloride': ['Cl-', 'chloride', 'chlorine'],	
'sulphate':['SO4-2', 'sulfate', 'sulfates', 'S', 'sulfide', 'sulfides', 'H2S', 'hydrogen sulfide', 'S2O3-2', 'thiosulfate'],		
'oxygen': ['O2', 'oxygen', 'oxidase'],	
'hydrogen': ['H2', 'hydrogen', 'hydrogenase', 'hydrogen uptake', 'hydrogen evolution'],	
'organics': ['methane', 'CH4', 'methanogenic', 'methanogenesis', 'formate','formic acid', 'HCOO-', 'acetate', 'acetic acid', 'CH3COO-', 'propionate','propionate', 'propionic acid', 'butyrate', 'butyric acid','lactate', 'lactic acid', 'mercaptans', 'mercaptan', 'thiol', 'methanethiol', 'ethanethiol', 'H2S', 'h2s', 'alcohol', 'ethanol', 'methanol', 'propanol', 'alcohol']
}		

corrosion_synergies= {
'Fe-S': ['iron_sulfur', 'Fe-S','iron sulfide','FeS', 'Fe-S cluster'],	
'Fe-Cl': ['iron chloride', 'FeCl', 'iron halide', 'ferric chloride'],	
'Fe-C': ['iron carbon', 'FeC', 'iron carbonate', 'siderite'],	
'Cu-Fe': ['copper iron', 'Cu-Fe', 'bimetallic', 'galvanic couple'],	
'Mn-Fe': ['manganese iron', 'Mn-Fe', 'iron manganese oxide'],
'Ni-Fe': ['Ni-Fe'],  	
'Cr-Fe': ['chromium iron', 'Cr-Fe', 'stainless steel', 'chromium passivation'], 	
'Al-Cu': ['aluminum copper', 'Al-Cu', 'aluminum brass', 'galvanic corrosion'], 	
'Zn-Fe': ['zinc iron', 'Zn-Fe', 'galvanized steel', 'sacrificial anode'],	
'Fe-CO3': ['iron carbonate', 'siderite', 'bicarbonate corrosion', 'carbonate scaling'],	
'Fe-SO4': ['iron sulfate', 'sulfate corrosion', 'gypsum formation'],	
'Fe-Ox': ['iron oxalate', 'oxalate corrosion', 'organic acid attack', 'oxidation corrosion'],	
'Fe-Ac': ['iron acetate', 'acetate corrosion', 'organic acid attack', 'oxidation corrosion']}	
	
functional_categories =	{
'o2_consumption': {'terms': ['o2_consumption', 'aerobic_respiration', 'oxygen reduction', 'oxygen consumption', 'cytochrome oxidase', 'oxidase', 'terminal oxidase',  'oxygen reductase',  'superoxide dismutase',  'catalase',  'oxidative stress', 'oxygen sensor',  'oxygen tolerance', 'oxygen consum', 'oxygen scavenging', 'oxygen stress', 'oxidative phosphorylation', 'NADH dehydrogenase'], 'score': 0.6, 'justification': ''},
'nitrogen_metabolism':  {'terms': ['nitrate_reduction', 'nitrite_reduction', 'denitrification', 'nitrification', 'nitrate respiration', 'nitrite respiration', 'nitrous oxide reduction', 'ammonia oxidation', 'anammox', 'nitrogen fixation', 'ammonification', 'nitrogen metabolism',  'nitrate',  'nitrite','dissimilatory nitrate reduction',  'nitrite reductase', 'nitrate reductase' ], 'score': 1.0, 'justification': ''},	
'iron_metabolism': {'terms': ['corrosion', 'MIC', 'ferric reduc', 'SRB', 'ocre', 'iron_oxide', 'iron_deposit', 'metal oxide', 'ochre formation', 'iron oxide deposits', 'iron precipitation', 'rust formation', 'iron oxid', 'ferrous oxid', 'ferric', 'iron uptake', 'iron transport', 'iron storage',  'iron homeostasis', 'siderophore production', 'iron_sulfur_redox', 'ferredoxin',  'rubredoxin', 'ferritin', 'bacterioferritin',  'PWY-7221',  'PWY-7219', 'HEME-BIOSYNTHESIS-II',  'P125-PWY', 'iron mobilization', 'iron immobilization', 'ferrihydrite', 'goethite', 'magnetite', 'hematite', 'iron mineral', 'biogenic iron oxides', 'stalactite formation', 'ochre mats'], 'score': 1.5, 'justification': ''},	
'sulfur_metabolism': {'terms': ['sulphur_metabolism', 'sulfur_metabolism', 'sulfate reduc', 'sulfite', 'thiosulfate', 'sulfur oxidation', 'SRB', 'dsrAB', 'APS reductase', 'sulfide','quinone oxidoreductase', 'dissimilatory sulfate reduction', 'sulfur globules', 'elemental sulfur', 'polysulfide metabolism', 'sulfur granules', 'PWY-6932', 'SO4ASSIM-PWY', 'SULFATE-CYS-PWY', 'sulfide_production', 'sulfonate',  'sulfur_reduction', 'desulfovibrio', 'sulfur disproportionation', 'sulfate-reducing bacteria', 'sulfur respiration'], 'score': 1.5, 'justification': ''},	
'h2_consumption': {'terms': ['h2_consumption', 'hydrogenase', 'hydrogen uptake', 'hydrogen consumption', 'h2', 'H2 oxidation', 'H2ase', 'hydrogen metabolism', 'hydrogen production',  'FeFe-hydrogenase',  'NiFe-hydrogenase',  'hydrogen evolution',  'hydrogen cycling',  'H2 sensing',  'proton reduction'], 'score': 0.5, 'justification': ''},	
'direct_eet':  {'terms': ['cytochrome c oxidase', 'quinol oxidase', 'NADH:quinone oxidoreductase', 'succinate dehydrogenase', 'fumarate reductase', 'cytochrome', 'electron transfer', 'electron transport', 'conductive pili', 'nanowire', 'mtrABC', 'omc', 'omcS', 'oxidoreductase', 'redox', 'reductase', 'oxidase', 'electron conduit', 'direct electron transfer', 'deet', 'c-type cytochrome', 'multi-heme cytochrome', 'flavin', 'electron shuttle',], 'score': 1.1, 'justification': ''},	
'carbon_metabolism': {'terms':  ['carbon_metabolism', 'carbon fixation', 'carbon utilization', 'carbohydrate metabolism', 'glycolysis', 'TCA cycle', 'carbon flux', 'carbon assimilation', 'pentose phosphate pathway', 'gluconeogenesis', 'Calvin cycle', 'reductive acetyl-CoA pathway', 'carbon monoxide dehydrogenase', 'GLYCOLYSIS', 'hydrocarbon degradation', 'aromatic degradation', 'alcohol metabolism', 'organic matter degradation', 'VFA production', 'propionate', 'butyrate', 'valerate', 'caproate'], 'score': 0.5, 'justification': ''},	
'indirect_eet': {'terms': ['shuttle', 'mediator', 'redox mediator', 'electron shuttle', 'flavin', 'quinone', 'humic substance'], 'score': 0.5, 'justification': ''},	
'organic_acid_metabolism': {'terms':  ['acetate', 'acetic acid', 'acetyl', 'acetate metabolism', 'acetate production', 'oxalate', 'oxalic acid', 'oxalate metabolism', 'oxalate production', 'organic acid', 'fatty acid', 'butyric acid', 'butyrate', 'propionate', 'propionic acid', 'carboxylic acid', 'lactate', 'lactic acid', 'formate', 'formic acid', 'citrate', 'citric acid', 'succinate', 'succinic acid', 'fumarate', 'fumaric acid', 'malate', 'malic acid', 'pyruvate', 'pyruvic acid', 'acidification', 'fermentation', 'CENTFERM-PWY', 'FERMENTATION-PWY', 'GLYCOLYSIS', 'PWY-5100', 'GALACTUROCAT-PWY'], 'score': 1.5, 'justification': ''},	
'metal binding / chelation': {'terms': ['metal_chelation', 'metal_binding', 'siderophore', 'complexation', 'iron chelation', 'enzymatic_metal_oxid', 'peroxidase',  'chelator', 'metallophore', 'iron complex', 'metal transport', 'metal oxide', 'iron oxide deposits', 'metal deposition', 'metal solubilization', 'mineral dissolution', 'mineral precipitation', 'chelation', 'metal complexation', 'metal sequestration' , 'metal_organic_interaction', 'metal organic', 'metal homeostasis', 'organometallic',  'iron uptake', 'metal uptake', 'metalloprotein',  'iron-sulfur cluster', 'metal coordination', 'ferric reductase', 'ferrous oxidase', 'metal homeostasis', 'mineral dissolution', 'mineral precipitation', 'copper reduction', 'nickel oxidation', 'chromium reduction', 'crystal nucleation', 'metal immobilization'], 'score': 1.0, 'justification': ''},	
'biofilm_formation': {'terms': ['biofilm_formation', 'metal_chelation', 'quorum_sensing', 'extracellular_matrix', 'exopolysaccharide', 'EPS production', 'EPS', 'surface_disruption', 'polysaccharide', 'adhesin', 'biofilm', 'EPS', 'extracellular polymeric substance', 'curli', 'exopolymer', 'extracellular matrix', 'adhesion', 'colonization', 'attachment', 'surface', 'adherence', 'biofilm maturation', 'biofilm regulation', 'biofilm dispersion', 'cell-cell adhesion',  'surface attachment', 'polysaccharide biosynthesis', 'cell aggregation', 'matrix production', 'pellicle', 'floc formation', 'COLANSYN-PWY', 'EXOPOLYSACC-PWY', 'GLUCOSE1PMETAB-PWY', 'alginate', 'cellulose', 'lipid metabolism', 'fatty acid synthesis', 'fatty acid degradation', 'biosurfactant', 'VFA', 'volatile fatty acid', 'propionate', 'butyrate', 'oleaginous', 'lipid accumulation', '3-oxoacyl'], 'score': 1.2, 'justification': ''},	
'manganese_processes': {'terms': ['manganese_reduction', 'mn_redox', 'manganese oxidation', 'manganese oxide',  'pyrolusite',  'birnessite',  'manganese cycling',  'manganese mineral',  'manganese transport', 'Mn-oxide formation', 'Mn-oxide reduction', 'Mn precipitation', 'Mn dissolution'], 'score': 1.0, 'justification': ''},	
'methanogenesis': {'terms': ['methanogenesis', 'methanobacterium', 'archaea', 'methane production', 'methyl-coenzyme M reductase', 'methanogenic', 'coenzyme F420',  'methyl-H4MPT', 'CO2 reduction', 'acetoclastic methanogenesis'], 'score': 0.6, 'justification': ''},	
'fumarate_formation': {'terms': ['fumarate', 'propionibacterium'], 'score': 0.5, 'justification': ''},	
'halogen_related': {'terms': ['halogen', 'chloride', 'bromide', 'iodide', 'fluoride', 'halide', 'dehalogenation', 'haloperoxidase', 'haloacid', 'chlorination', 'bromination', 'organohalide', 'halomethane', 'haloalkane', 'organohalide', 'halotolerance', 'salt tolerance', 'halophilic', 'chloride transport', 'halide channel', 'chloride attack', 'chloride-induced corrosion', 'pitting initiation', 'chloride penetration', 'halide corrosion', 'perchlorate reduction', 'halorespiration', 'organohalide'], 'score': 0.7, 'justification': ''},	
'ph_modulation': {'terms': ['acid', 'alkaline', 'proton pump', 'pH homeostasis', 'pH stress', 'acid tolerance', 'alkaline tolerance', 'proton motive force', 'pH regulation', 'acidic environment', 'alkaline environment', 'acid resistance', 'proton antiporter', 'proton generation', 'low pH', 'pH buffering', 'pH gradient', 'urease', 'ammonification', 'ammonia production', 'alkali production'], 'score': 0.5, 'justification': ''},	
'phosphorus_metabolism': {'terms': ['phosphate transport',  'polyphosphate', 'phosphite oxidation', 'organophosphonate metabolism'], 'score': 0.5, 'justification': ''},	
'mic' : {'terms': ['antimicrobial production', 'competitive exclusion', 'corrosion inhibition', 'fungal metabolism', 'archaeal metabolism', 'extremophile'], 'score': 0.5, 'justification': ''},	
'temp_response': {'terms':['heat shock', 'cold shock', 'temperature response', 'thermophilic', 'psychrophilic', 'mesophilic', 'thermal adaptation', 'temperature stress', 'heat stress protein','cold stress protein', 'thermal stability', 'thermotolerance' ,'osmotic stress', 'desiccation tolerance'],'score': 0.2, 'justification': ''},
'enzymatic_metal_oxid': {'terms': ['metalloenzyme', 'enzyme-catalyzed oxidation', 'peroxidase', 'laccase', 'oxidoreductase activity', 'enzyme-mediated corrosion'], 'score': 0.8, 'justification': ''},
'dealloying_mechanisms': {'terms': ['selective corrosion', 'dezincification', 'dealuminification', 'preferential dissolution', 'parting'], 'score': 0.6, 'justification': ''},
'exoelectrogenesis': {'terms': ['exoelectrogen', 'electrochemically active bacteria', 'EAB', 'extracellular respiration', 'electrode respiration'], 'score': 0.9, 'justification': ''}
}

In [ ]:
#df = ECcontri_Uniprot_enriched.sample(n=1500)

In [21]:
text_columns = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
                'corrosion_mechanisms', 'functional_categories', 'corrosion_keyword_groups', 
                'corrosion_synergies', 'organic_processes', 'pathways', 'ipath', 'reactions']

analyzer = BiologicalTermDiscovery()
new_terms = analyzer.discover_novel_terms(df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns)
print(new_terms)

['acid', 'acids', 'alcoholic', 'alkaloid', 'alkyl', 'amoebiasis', 'antimicrobial', 'cell', 'cellular', 'ch-oh', 'chromosome', 'coli', 'cycle', 'disease', 'enzymes', 'ester', 'exosome', 'glycerophospholipid', 'group', 'lipopolysaccharide', 'mannose', 'membrane', 'metabolic', 'metabolism', 'metabolites', 'methyl', 'molecular', 'molecules', 'nitrogen', 'nitrogen_metabolism', 'nitrogenous', 'nucleotide', 'numbers', 'oxygenases', 'peptide', 'peroxide', 'polycyclic', 'polyketide', 'polyketides', 'processes', 'prokaryotes', 'prokaryotic', 'resistance', 'ribosome', 'selenocompound', 'sucrose', 'sulfur', 'syndrome', 'transfer', 'tropane', 'unclassified', 'various']


# Dictionary Refinement

**Dictionary Refinement and Validation for Corrosion-Related Functional Annotation**
To support downstream analysis and scoring in our corrosion microbiome framework, we constructed and iteratively refined a master annotation dictionary (global_terms) composed of multiple biologically meaningful categories (e.g., metal_terms, corrosion_mechanisms, pathway_categories, functional_categories). These dictionaries initially included a comprehensive, theory-driven list of potential terms derived from domain knowledge and literature.

However, during integration with bioinformatic annotation data, it became clear that many terms lacked empirical support in our dataset. To resolve this, we implemented a data-driven refinement workflow as follows:

Step 1: Validation of Terms Against Enriched Data
A custom validation script compared the theoretical dictionary entries against the actual protein annotation fields (e.g., EC numbers, enzyme names, pathways) from our enriched dataset. This yielded a list of real_terms — annotation terms empirically supported in our system.

Step 2: Hierarchical Consolidation of Dictionary Structure
The global_terms structure was then refined using a hybrid strategy:

Minimum viable subcategories: Subcategories (e.g., specific corrosion mechanisms like direct_eet or galvanic_corrosion) were only retained if they contained sufficient real-world support. Sparse subcategories with low or no empirical evidence were eliminated or merged to reduce fragmentation.

Maximum term depth: Within each retained subcategory, we aimed to maximize the number of valid child terms (i.e., biological keywords, gene or enzyme names) to ensure rich annotation coverage.

Semantic reallocation: Unused terms from deprecated categories such as organic_processes and corrosion_keyword_groups were manually reclassified into valid categories where conceptually appropriate (e.g., terms like quorum sensing were reassigned to functional_categories).

This balance of data-driven filtering and semantic grouping led to a revised version of global_terms, maintaining a biologically coherent structure while aligning with the annotation reality of our dataset.

Step 3: Scoring Category Reduction
For downstream scoring and modeling, we focused only on three high-confidence, high-coverage categories:

metal_terms

corrosion_synergies

functional_categories

Other categories (e.g., corrosion_mechanisms, pathway_categories) were retained for network analysis and visualization, but excluded from scoring due to redundancy or sparsity.

# smart_consolidate_terms

In [ ]:
def smart_consolidate_terms(global_terms_list, real_terms):
    """
    Match real_terms to global_terms structure.
    - Retains global_terms structure.
    - Adds unmatched but valid terms to the correct top-level category, in a 'miscellaneous' subcategory.
    - Keeps functional_category scores and justification intact.
    - Collects truly unrecognized terms in manual_review.
    
    Args:
        global_terms_list: List of tuples like [('metal_terms', metal_dict), ...]
        real_terms: Dict {subcat: [terms]} from validation script

    Returns:
        consolidated: Updated global_terms-like dict with only valid real_terms
    """
    from collections import defaultdict
    import copy

    # Start from a deep copy of the base terms
    consolidated = {
        'metal_terms': defaultdict(list),
        'corrosion_synergies': defaultdict(list),
        'functional_categories': defaultdict(lambda: {'terms': [], 'score': 1.0}),
        'corrosion_mechanisms': defaultdict(list),
        'pathway_categories': defaultdict(list),
        'manual_review': defaultdict(list)
    }

    # Preserve all subcategories from global_terms
    term_index = {}  # term_lower → (cat, subcat, score)

    for category_name, cat_dict in global_terms_list:
        for subcat, value in cat_dict.items():
            if isinstance(value, dict) and 'terms' in value:
                score = value.get('score', 1.0)
                for term in value['terms']:
                    term_index[term.lower()] = (category_name, subcat, score)
                    consolidated[category_name][subcat] = {
                        'terms': copy.deepcopy(value['terms']),
                        'score': score,
                        'justification': value.get('justification', '')
                    }
            elif isinstance(value, list):
                for term in value:
                    term_index[term.lower()] = (category_name, subcat, None)
                consolidated[category_name][subcat] = copy.deepcopy(value)

    # Reallocate real terms into this structure
    for subcat, terms in real_terms.items():
        for term in terms:
            tkey = term.lower()
            if tkey in term_index:
                cat, existing_subcat, score = term_index[tkey]

                if cat == 'functional_categories':
                    if term not in consolidated[cat][existing_subcat]['terms']:
                        consolidated[cat][existing_subcat]['terms'].append(term)
                else:
                    if term not in consolidated[cat][existing_subcat]:
                        consolidated[cat][existing_subcat].append(term)

            else:
                # Try to infer best category
                likely_cat = (
                    'functional_categories' if 'ase' in tkey or 'eet' in tkey else
                    'metal_terms' if any(m in tkey for m in ['fe', 'cu', 'zn', 'mn']) else
                    'corrosion_synergies' if '-' in tkey else
                    'pathway_categories', 'corrosion_mechanisms'
                )

                if likely_cat == 'functional_categories':
                    consolidated[likely_cat]['miscellaneous']['terms'].append(term)
                elif likely_cat == 'metal_terms':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'corrosion_synergies':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'pathway_categories':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'corrosion_mechanisms':
                    consolidated[likely_cat]['corrosion_mechanisms'].append(term)
                else:
                    consolidated['manual_review'][subcat].append(term)

    return consolidated


In [ ]:
consolidated = smart_consolidate_terms([
    ('metal_terms', cs.metal_terms),
    ('corrosion_mechanisms', cs.corrosion_mechanisms),
    ('pathway_categories', cs.pathway_categories),
    ('corrosion_synergies', cs.corrosion_synergies),
    ('functional_categories', cs.functional_categories)
], real_terms)



In [ ]:
consolidated

{'metal_terms': defaultdict(list,
             {'iron': ['Fe2+',
               'Fe3+',
               'iron',
               'ferrous',
               'ferric',
               'heme',
               'iron-sulfur',
               'rust',
               'ochre',
               'iron oxide',
               'iron precipitation',
               'siderophore',
               'ferritin'],
              'manganese': ['Mn2+',
               'manganese',
               'mn',
               'manganous',
               'manganic',
               'manganese oxidation',
               'manganese oxide',
               'MnO2'],
              'copper': ['Cu+',
               'Cu2+',
               'copper',
               'cupric',
               'cuprous',
               'copper oxide',
               'copper corrosion'],
              'nickel': ['Ni2+',
               'nickel',
               'nickelous',
               'nickel oxidation',
               'nickel reduction'],
              'cobalt':